In [2]:
import re
import pandas as pd
import unicodedata
from pathlib import Path

# ---------- PREPOSITIONS ----------
PREPOSITIONS = [
    r"\bde la\b",
    r"\bde\b",
    r"\bdu\b",
    r"\bdes\b",
    r"\bd’\b",
    r"\bd'un\b",
    r"\bd'une\b",
    r"\bde l’\b"
]

pattern = re.compile(
    rf"(.+?)\s+(?:{'|'.join(PREPOSITIONS)})\s+(.+?)$",
    flags=re.IGNORECASE
)

# ---------- TEXT CLEANING ----------
def nettoyer_texte(s: str) -> str:
    if not s:
        return None
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\d+", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# ---------- PAIR EXTRACTION ----------
def extraire_paire(ligne: str):
    m = pattern.match(ligne)
    if not m:
        return None
    mot1, mot2 = map(nettoyer_texte, m.groups())
    if not mot1 or not mot2:
        return None
    return mot1, mot2

# ---------- MAIN PIPELINE ----------
def fusionner_fichiers(input_dir: str, output_csv: str):
    resultats = []

    for path in Path(input_dir).glob("*.txt"):
        relation = path.stem  # filename without .txt
        print(f"📄 Processing {relation}")

        with open(path, encoding="utf-8") as f:
            for ligne in f:
                ligne = ligne.strip()
                if not ligne:
                    continue
                paire = extraire_paire(ligne)
                if paire:
                    mot1, mot2 = paire
                    resultats.append({
                        "mot1": mot1,
                        "mot2": mot2,
                        "relation": relation
                    })

    df = pd.DataFrame(resultats)
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)
    df.to_csv(output_csv, index=False, encoding="utf-8")

    print(f"\n✅ {len(df)} total relations written to {output_csv}")

# ---------- RUN ----------
if __name__ == "__main__":
    fusionner_fichiers(
        input_dir="data",
        output_csv="dataset_global.csv"
    )


📄 Processing r_topic
📄 Processing r_object_matière
📄 Processing r_depict
📄 Processing r_product_of
📄 Processing r_processusinstr-1
📄 Processing r_own-1
📄 Processing r_processusagent
📄 Processing r_holo
📄 Processing r_has_causitif
📄 Processing r_has_property
📄 Processing r_lieu_origine
📄 Processing r_social_tie
📄 Processing r_quantificateur
📄 Processing r_processuspatient

✅ 10884 total relations written to dataset_global.csv
